# 06 · Joker

Assigns median household income to each establishment using US Census Bureau data.

For each shop's lat/lon, the Census geocoder identifies the census tract, then the ACS 5-year estimates provide median household income (table B19013).

**Input:** `csv/00_base_data.csv`
**Output:** `csv/06_joker.csv` — `osm_id`, `median_income`

In [ ]:
import pandas as pd
import requests
import time

df = pd.read_csv("csv/00_base_data.csv")
print(f"Loaded {len(df)} records")

In [ ]:
# ── Step 1: Map each unique lat/lon to its census tract ──

def get_census_tract(lat, lon):
    """Uses Census Bureau geocoder to find the FIPS tract for a coordinate."""
    url = "https://geocoding.geo.census.gov/geocoder/geographies/coordinates"
    params = {
        "x": lon, "y": lat,
        "benchmark": "Public_AR_Current",
        "vintage": "Current_Current",
        "format": "json",
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        geographies = r.json()["result"]["geographies"]
        tracts = geographies.get("Census Tracts", [])
        if tracts:
            t = tracts[0]
            return t["STATE"], t["COUNTY"], t["TRACT"]
    except Exception:
        pass
    return None, None, None


# Deduplicate coordinates to minimize API calls
# Round to 4 decimal places (~11 m precision) to group nearby shops
df["_lat_r"] = df["lat"].round(4)
df["_lon_r"] = df["lon"].round(4)
unique_coords = df[["_lat_r", "_lon_r"]].drop_duplicates()
print(f"Unique coordinate groups: {len(unique_coords)} (from {len(df)} shops)")

# Geocode each unique coordinate
tract_cache = {}
for i, (_, row) in enumerate(unique_coords.iterrows()):
    key = (row["_lat_r"], row["_lon_r"])
    if key not in tract_cache:
        state, county, tract = get_census_tract(row["_lat_r"], row["_lon_r"])
        tract_cache[key] = (state, county, tract)
        time.sleep(0.15)  # rate limit
    if (i + 1) % 100 == 0:
        print(f"  Geocoded {i+1}/{len(unique_coords)} coordinates")

# Map tracts back to shops
df["_state"]  = df.apply(lambda r: tract_cache.get((r["_lat_r"], r["_lon_r"]), (None,None,None))[0], axis=1)
df["_county"] = df.apply(lambda r: tract_cache.get((r["_lat_r"], r["_lon_r"]), (None,None,None))[1], axis=1)
df["_tract"]  = df.apply(lambda r: tract_cache.get((r["_lat_r"], r["_lon_r"]), (None,None,None))[2], axis=1)

print(f"\nTracts found: {df['_tract'].notna().sum()}/{len(df)}")
print(f"Unique tracts: {df['_tract'].nunique()}")

In [ ]:
# ── Step 2: Get median income per tract from ACS ──────

def get_acs_median_income(state, county, tract):
    """Fetches median household income (B19013_001E) from ACS 5-year estimates."""
    url = "https://api.census.gov/data/2022/acs/acs5"
    params = {
        "get": "B19013_001E",
        "for": f"tract:{tract}",
        "in": f"state:{state} county:{county}",
    }
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        if len(data) > 1:
            val = data[1][0]
            if val and val not in ["-666666666", "-222222222"]:
                return int(val)
    except Exception:
        pass
    return None


# Fetch income for each unique tract
unique_tracts = df[["_state", "_county", "_tract"]].dropna().drop_duplicates()
print(f"Fetching income for {len(unique_tracts)} unique tracts...")

income_cache = {}
for i, (_, row) in enumerate(unique_tracts.iterrows()):
    key = (row["_state"], row["_county"], row["_tract"])
    income = get_acs_median_income(*key)
    income_cache[key] = income
    time.sleep(0.1)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(unique_tracts)} tracts")

# Map income back to shops
df["median_income"] = df.apply(
    lambda r: income_cache.get((r["_state"], r["_county"], r["_tract"]), None),
    axis=1
)

print(f"\nIncome fill: {df['median_income'].notna().sum()}/{len(df)}")
print(f"Income range: ${df['median_income'].min():,.0f} - ${df['median_income'].max():,.0f}")

In [ ]:
df_out = df[["osm_id", "median_income"]]
df_out.to_csv("csv/06_joker.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/06_joker.csv")
print(df_out.describe().round(0))